# 05 回归训练：基线与最终方案

Edited by Wang Hongjie

同一批样本和 6 项特征比较 XGBoost 与岭回归。目标为 t+1 至 t+2 开盘收益率。

回归任务： 预测实际收益率。
$$
r=\frac{Open_{t+2}}{Open_{t+1}}-1
$$

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import hashlib
import json
import logging
import os
import platform
import shutil
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import sklearn
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, r2_score, roc_auc_score
from IPython.display import display

# Run from SP500; all notebook helpers are local.
ROOT = Path.cwd().resolve()
if not (ROOT / "01_data_preprocessing.ipynb").is_file():
    raise FileNotFoundError("Run this notebook from the SP500 directory.")


### 路径与特征名称

配置本文件使用的数据位置和特征列表。


In [2]:
DATA_DIR = ROOT / "data"

CLEAN_FILE = DATA_DIR / "processed" / "sp500_clean.csv"

FEATURE_DIR = DATA_DIR / "features"

RAW_FILE = DATA_DIR / "raw" / "sp500_daily_1996-11-13_to_2026-11-12_raw.json"

ORIGINAL = [
    "return_1d", "return_5d", "return_20d", "volatility_20d",
    "intraday_return", "intraday_range",
]

BP = 10_000


### 表格、图像与文件保存

保存 CSV/JSON，并在 Notebook 中展示结果。


In [3]:
def display_path(path):
    """Show relative project paths and external filenames."""
    path = Path(path)
    if not path.is_absolute():
        return path.as_posix()
    try:
        return Path(os.path.relpath(path, ROOT)).as_posix()
    except ValueError:
        return path.name

def save_json(path, value):
    """Write UTF-8 JSON without nonstandard NaN or Infinity."""
    def convert(item):
        if isinstance(item, np.generic):
            return item.item()
        if isinstance(item, (Path, pd.Timestamp, datetime)):
            return str(item)
        raise TypeError(f"Cannot serialize {type(item)}")
    Path(path).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=convert, allow_nan=False),
        encoding="utf-8",
    )

def save_csv(path, frame, index=True):
    frame.to_csv(path, index=index, encoding="utf-8-sig", index_label="signal_date" if index else None)

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def show_table(frame, decimals=4):
    """Display rounded values without changing saved data."""
    display(frame.round(decimals))


### 读取上游数据与构造标签

读取已经准备的数据；标签与跨市场特征按原时间约定构造。


In [4]:
def load_prepared_samples(task):
    """Load step 02 samples and verify upstream inputs."""
    if task not in ("classification", "regression"):
        raise ValueError("Unknown task.")
    path = FEATURE_DIR / f"{task}_samples.csv"
    manifest_file = FEATURE_DIR / "feature_build.json"
    if not path.is_file() or not manifest_file.is_file():
        raise FileNotFoundError("Run steps 01 and 02 first.")
    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
    for source in manifest["sources"]:
        if file_hash(ROOT / source["path"]) != source["sha256"]:
            raise ValueError("Upstream inputs changed; rerun step 02.")
    if file_hash(path) != manifest["outputs"][path.name]:
        raise ValueError("Feature file changed; rerun step 02.")
    samples = pd.read_csv(path, index_col="signal_date",
                          parse_dates=["signal_date", "trade_date", "settle_date"],
                          float_precision="round_trip")
    samples["target_class"] = samples["target_class"].astype("Int8")
    assert samples.index.is_monotonic_increasing and samples.index.is_unique
    return samples


### 按时间划分并剔除越界标签

沿用有效样本的 8:1:1 边界，剔除跨分区收益标签。

In [5]:
def chronological_split(samples, ratios=(0.8, 0.1, 0.1)):
    """Split valid samples by date; drop targets crossing boundaries."""
    if len(ratios) != 3 or not np.isclose(sum(ratios), 1) or min(ratios) <= 0:
        raise ValueError("Provide three positive ratios summing to one.")
    val_size, test_size = (round(len(samples) * x) for x in ratios[1:])
    train_size = len(samples) - val_size - test_size
    if min(train_size, val_size, test_size) < 2:
        raise ValueError("Too few samples in a split.")
    validation_start = samples.iloc[train_size]["trade_date"]
    test_start = samples.iloc[train_size + val_size]["trade_date"]
    splits = {
        "train": samples.loc[samples.settle_date < validation_start].copy(),
        "val": samples.loc[(samples.trade_date >= validation_start) & (samples.settle_date < test_start)].copy(),
        "test": samples.loc[samples.trade_date >= test_start].copy(),
    }
    assert all(len(frame) > 0 for frame in splits.values())
    assert splits["train"].settle_date.max() <= splits["val"].index.min()
    assert splits["val"].settle_date.max() <= splits["test"].index.min()
    all_dates = pd.Index(np.concatenate([f.index.to_numpy() for f in splits.values()]))
    assert all_dates.is_unique, "Duplicate signal dates across splits."
    summary = pd.DataFrame([
        {"split": name, "samples": len(frame), "first_signal": frame.index.min(),
         "last_signal": frame.index.max(), "first_entry": frame.trade_date.min(),
         "last_settlement": frame.settle_date.max()}
        for name, frame in splits.items()
    ])
    return splits, summary


### 实验目录与逐轮训练日志

每次运行新建目录，保存配置、数据指纹、代码快照和逐轮日志。


In [6]:
@dataclass
class Run:
    path: Path
    config: dict
    logger: logging.Logger

def start_run(name, config, sources):
    """Create a new run directory without overwriting results."""
    stamp = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%d_%H%M%S_%f")
    path = ROOT / "output" / name / stamp
    path.mkdir(parents=True, exist_ok=False)
    logger = logging.getLogger(f"{name}.{stamp}")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    # Keep the notebook default warning format.
    fmt = logging.Formatter("%(asctime)s | %(message)s", datefmt="%H:%M:%S")
    for handler in (logging.FileHandler(path / "run.log", encoding="utf-8"), logging.StreamHandler(sys.stdout)):
        handler.setFormatter(fmt)
        logger.addHandler(handler)
    config = dict(config, scenario=name, created_at=datetime.now(ZoneInfo("Asia/Singapore")).isoformat())
    save_json(path / "config.json", config)
    save_json(path / "environment.json", {
        "python": platform.python_version(), "python_executable": sys.executable,
        "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__, "timezone": "Asia/Singapore",
    })
    save_json(path / "data_sources.json", [
        {"path": str(p), "sha256": file_hash(p), "bytes": p.stat().st_size}
        for p in sources
    ])
    snapshot = path / "code"
    snapshot.mkdir()
    for source in sorted(ROOT.glob("0[1-7]_*.ipynb")):
        if source.exists():
            shutil.copy2(source, snapshot / source.name)
    logger.info("Started %s; output directory %s", name, display_path(path))
    return Run(path, config, logger)

def stage_log(run, directory, message):
    directory.mkdir(parents=True, exist_ok=True)
    message = str(message)
    run.logger.info(message)
    with (directory / "stage.log").open("a", encoding="utf-8") as stream:
        stream.write(f"{datetime.now().isoformat(timespec='seconds')} | {message}\n")

class TrainingLog(xgb.callback.TrainingCallback):
    """Write each XGBoost round to the display and run logs."""
    def __init__(self, run, directory, every=1):
        self.run, self.directory, self.every = run, directory, every

    def after_iteration(self, model, epoch, evals_log):
        if epoch == 0 or (epoch + 1) % self.every == 0:
            values = [f"{part}.{metric}={scores[-1]:.8f}"
                      for part, metrics in evals_log.items() for metric, scores in metrics.items()]
            stage_log(self.run, self.directory, f"Round {epoch + 1} | " + " | ".join(values))
        return False

class LoggedEarlyStopping(xgb.callback.EarlyStopping):
    """Log metrics before checking early stopping, including the final round."""
    def __init__(self, run, directory, log_every=1, **kwargs):
        super().__init__(**kwargs)
        self.progress = TrainingLog(run, directory, every=log_every)

    def after_iteration(self, model, epoch, evals_log):
        self.progress.after_iteration(model, epoch, evals_log)
        return super().after_iteration(model, epoch, evals_log)


### 模型保存、重新加载与指标计算

保存训练和验证结果，测试时重新加载模型，并按原定义计算指标。


In [7]:
def predict_model(model, frame, columns, unit):
    if isinstance(model, xgb.Booster):
        result = model.predict(xgb.DMatrix(frame[columns], nthread=4))
    else:
        result = model.predict(frame[columns])
    return result / BP if unit == "basis_points" else result

def measure(task, actual, prediction, threshold=None):
    if task == "classification":
        decisions = prediction >= threshold
        return {
            "roc_auc": roc_auc_score(actual, prediction),
            "ap": average_precision_score(actual, prediction),
            "accuracy": accuracy_score(actual, decisions),
            "balanced_accuracy": balanced_accuracy_score(actual, decisions),
            "f1": f1_score(actual, decisions, zero_division=0),
        }
    error = (prediction - actual) * BP
    return {"mae_bp": np.abs(error).mean(), "rmse_bp": np.sqrt(np.mean(error ** 2)), "r2": r2_score(actual, prediction)}

def new_training_dir(run, period):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    parent = run.path if period == "holdout" else run.path / "folds" / period
    folder = parent / f"train_{stamp}"
    folder.mkdir(parents=True, exist_ok=False)
    return folder

def save_training(run, period, folder, model, splits, columns, selected, history=None):
    """Save model and train/validation predictions before testing."""
    folder.mkdir(parents=True, exist_ok=True)
    task = run.config["task"]
    selected = dict(selected, task=task, features=list(columns),
                    train_mean=float(splits["train"].target_return.mean()),
                    train_median=float(splits["train"].target_return.median()))
    save_json(folder / "selected.json", selected)
    if isinstance(model, xgb.Booster):
        model.save_model(folder / "model.ubj")
    else:
        joblib.dump(model, folder / "model.joblib")
    if history:
        save_json(folder / "history.json", history)
        history_frame = pd.DataFrame({f"{part}_{metric}": values
                                      for part, metrics in history.items() for metric, values in metrics.items()})
        history_frame.index += 1
        history_frame.index.name = "round"
        history_frame.to_csv(folder / "history.csv", encoding="utf-8-sig")
    label = "target_class" if task == "classification" else "target_return"
    metrics = {}
    for split in ("train", "val"):
        frame = splits[split]
        prediction = predict_model(model, frame, columns, selected["prediction_unit"])
        output = frame[["trade_date", "settle_date", label]].assign(prediction=prediction)
        save_csv(folder / f"{split}_predictions.csv", output)
        metrics[split] = measure(task, frame[label].to_numpy(dtype=float), prediction, selected.get("threshold"))
    save_json(folder / "metrics.json", metrics)
    stage_log(run, folder, f"{period} training complete | selected {selected.get('alpha', selected.get('best_rounds'))}")
    return {"folder": folder, "selected": selected}

def evaluate_saved(run, period, artifact, test):
    """Reload the model and create a fresh test directory."""
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    parent = run.path if period == "holdout" else run.path / "folds" / period
    folder = parent / f"test_{stamp}"
    folder.mkdir(parents=True, exist_ok=False)
    selected = artifact["selected"]
    source = artifact["folder"]
    if (source / "model.ubj").exists():
        model = xgb.Booster(params={"nthread": 4})
        model.load_model(source / "model.ubj")
    else:
        model = joblib.load(source / "model.joblib")
    columns, task = selected["features"], selected["task"]
    stage_log(run, folder, f"{period} testing started | samples {len(test)} | model loaded from disk")
    prediction = predict_model(model, test, columns, selected["prediction_unit"])
    if not np.isfinite(prediction).all():
        raise ValueError("Model produced invalid predictions.")
    label = "target_class" if task == "classification" else "target_return"
    actual = test[label].to_numpy(dtype=float)
    metrics = {"period": period, "n_test": len(test), **measure(task, actual, prediction, selected.get("threshold"))}
    predictions = test[["trade_date", "settle_date", label]].assign(prediction=prediction)
    if task == "classification":
        predictions["predicted_class"] = (prediction >= selected["threshold"]).astype(int)
        metrics.update(threshold=selected["threshold"], positive_rate=float(actual.mean()))
        save_csv(folder / "confusion_matrix.csv", pd.DataFrame(
            confusion_matrix(actual, predictions.predicted_class, labels=[0, 1]),
            columns=["predicted_0", "predicted_1"]), index=False)
    else:
        # Use train median for MAE and train mean for RMSE baselines.
        predictions["constant_mean"] = selected["train_mean"]
        predictions["constant_median"] = selected["train_median"]
        metrics["constant_mae_bp"] = np.abs(actual - selected["train_median"]).mean() * BP
        metrics["constant_rmse_bp"] = np.sqrt(np.mean((actual - selected["train_mean"]) ** 2)) * BP
        metrics["delta_mae_bp"] = metrics["mae_bp"] - metrics["constant_mae_bp"]
        metrics["delta_rmse_bp"] = metrics["rmse_bp"] - metrics["constant_rmse_bp"]
    save_csv(folder / "predictions.csv", predictions)
    save_json(folder / "metrics.json", metrics)
    save_json(folder / "model_source.json", {"training_directory": str(source), "selected": selected})
    stage_log(run, folder, f"{period} testing complete | " + " | ".join(
        f"{key}={metrics[key]:.4f}" for key in (("roc_auc", "ap") if task == "classification" else ("mae_bp", "rmse_bp"))))
    return metrics, predictions


## 1. 导入与配置


In [8]:
# Reuse regression tree settings; select XGBoost rounds by MAE and ridge alpha by RMSE.
# Keep the original search range.
XGB_PARAMS = {
    "tree_method": "hist",       # Histogram tree growth for daily data
    "max_depth": 3,              # Limit tree complexity
    "eta": 0.03,                 # Per-round learning rate
    "min_child_weight": 5,
    "subsample": 0.8,            # Sample training rows each round
    "colsample_bytree": 0.9,     # Sample features for each tree
    "lambda": 5.0,               # L2 regularization
    "seed": 42, "nthread": 4,
}
MAX_ROUNDS = 400
EARLY_STOPPING = 30              # Stop after 30 validation rounds without improvement

ALPHAS = [0.1, 10, 100, 1_000, 10_000]

SCENARIO = "regression_comparison"
ARM_LABELS = {'baseline': 'Baseline · 6 features + XGBoost', 'final': 'Final · 6 features + Ridge'}
FEATURE_SETS = {"baseline": ORIGINAL, "final": ORIGINAL}
CONFIG = {
    "task": "regression", "evaluation": "single_chronological_holdout",
    "split_ratios": [0.8, 0.1, 0.1], "horizon_trading_days": 1,
    "positive_threshold": 0.002, "label": "open[t+2] / open[t+1] - 1",
    "last_data_date": "2026-11-12", "seed": 42,
    "tree_params": XGB_PARAMS, "max_rounds": MAX_ROUNDS, "early_stopping": EARLY_STOPPING,
}
CONFIG["ridge_alpha_candidates"] = ALPHAS  # Compare regularization strengths on validation only
RUN = start_run(SCENARIO, CONFIG, [RAW_FILE, CLEAN_FILE, FEATURE_DIR / "regression_samples.csv", FEATURE_DIR / "feature_build.json"])
arm_runs = {}
for arm in ARM_LABELS:
    folder = RUN.path / arm
    folder.mkdir()
    arm_config = dict(CONFIG, arm=ARM_LABELS[arm], features=FEATURE_SETS[arm])
    save_json(folder / "config.json", arm_config)
    arm_runs[arm] = Run(folder, arm_config, RUN.logger)


20:09:39 | Started regression_comparison; output directory output/regression_comparison/20261124_200939_835752


## 2. 特征与标签

两组共用 6 项历史价格特征；目标为 `open[t+2]/open[t+1]-1`。


In [9]:
samples = load_prepared_samples("regression")  # Verify upstream file hashes before training
save_csv(RUN.path / "features_and_targets.csv", samples)
show_table(samples[ORIGINAL + ["target_return", "target_class"]].tail(3))


,return_1d,return_5d,return_20d,volatility_20d,intraday_return,intraday_range,target_return,target_class
signal_date,,,,,,,,
2026-11-06,0.0031,-0.0268,-0.0556,0.0125,-0.0001,0.0101,0.0122,1
2026-11-09,0.0235,0.0011,-0.0330,0.0138,0.0172,0.0209,0.0098,1
2026-11-10,-0.0038,-0.0063,-0.0365,0.0138,0.0010,0.0098,-0.0047,0


## 3. 单次按时间划分
按时间顺序将数据划分为训练集 : 验证集 : 测试集 -- 80% : 10% : 10%。两种方案使用相同的数据划分。


In [10]:
# Pass original ratios to the chronological splitter.
splits, split_summary = chronological_split(samples, CONFIG["split_ratios"])  # Remove boundary-crossing labels
save_csv(RUN.path / "split_summary.csv", split_summary, index=False)
show_table(split_summary)
RUN.logger.info("Chronological split: train %s / validation %s / test %s",
                *(len(splits[k]) for k in ("train", "val", "test")))


C:\Users\w\AppData\Local\Temp\ipykernel_52040\1163363144.py:32: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(frame.round(decimals))


,split,samples,first_signal,last_signal,first_entry,last_settlement
0,train,6020,1996-12-12,2020-11-11,1996-12-13,2020-11-13
1,val,751,2020-11-13,2023-11-08,2020-11-16,2023-11-10
2,test,752,2023-11-10,2026-11-10,2023-11-13,2026-11-12


20:09:42 | Chronological split: train 6020 / validation 751 / test 752


## 4. 定义 XGBoost 训练与调参函数
保留历史基线的平方误差训练、验证 MAE 选轮次设置。函数只接收训练和验证数据。


In [11]:
def train_xgboost(run, stage, train, val, columns):
    """Tune on training and validation data only."""
    folder = new_training_dir(run, stage)
    stage_log(run, folder, f"{stage} XGBoost training started | train={len(train)} / val={len(val)}")
    label = "target_return"
    dtrain = xgb.DMatrix(train[columns], label=train[label], nthread=4)
    dval = xgb.DMatrix(val[columns], label=val[label], nthread=4)
    params = dict(XGB_PARAMS, objective="reg:squarederror", eval_metric="mae")
    history = {}

    # Use a fresh callback; keep trees through the best validation round.
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=MAX_ROUNDS,
        evals=[(dtrain, "train"), (dval, "val")],
        evals_result=history,
        callbacks=[
            # Log each round before the early-stop check.
            LoggedEarlyStopping(run, folder, log_every=1,
                rounds=EARLY_STOPPING, metric_name="mae",
                data_name="val", maximize=False, save_best=True,
            ),
        ],
        verbose_eval=False,
    )
    selected = {
        "params": params, "best_rounds": int(model.best_iteration + 1),
        "trained_rounds": len(history["val"]["mae"]),
        "validation_metric": "mae", "validation_score": float(model.best_score),
        "prediction_unit": "return",
        "training_window": "Same fixed training set",
    }

    # Retain the original squared-error training and MAE-based rounds.
    # Use RMSE to tune and evaluate the final regression arm.
    stage_log(run, folder, f"Best validation MAE round {selected['best_rounds']}")

    stage_log(run, folder, f"Trained {selected['trained_rounds']} rounds; retained best {selected['best_rounds']}-round model")
    return save_training(run, stage, folder, model, {"train": train, "val": val},
                         columns, selected, history)


## 5. 定义岭回归调参函数
标准化只拟合训练集；在同一验证集上比较 5 个 alpha，按 RMSE 选择最佳者。


In [12]:
def train_ridge(run, stage, train, val, columns):
    """Select alpha by validation RMSE on the fixed train split."""
    folder = new_training_dir(run, stage)
    stage_log(run, folder, f"{stage} Ridge tuning started | train={len(train)} / val={len(val)}")
    candidates, fitted = [], []
    for alpha in ALPHAS:
        candidate_dir = folder / f"alpha_{alpha:g}"
        stage_log(run, candidate_dir, f"Fitting alpha={alpha:g}")
        # Pipeline fits scaling on train only and reuses it for validation/test.
        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        model.fit(train[columns], train["target_return"] * BP)
        validation_prediction_bp = model.predict(val[columns])
        error_bp = validation_prediction_bp - val["target_return"].to_numpy() * BP
        rmse_bp = float(np.sqrt(np.mean(error_bp ** 2)))
        candidates.append({"alpha": alpha, "validation_rmse_bp": rmse_bp})
        fitted.append(model)
        # Save each candidate model, predictions, and logs.
        joblib.dump(model, candidate_dir / "model.joblib")
        save_json(candidate_dir / "metrics.json", candidates[-1])
        save_csv(candidate_dir / "validation_predictions.csv", val[
            ["trade_date", "settle_date", "target_return"]
        ].assign(prediction=validation_prediction_bp / BP))
        stage_log(run, candidate_dir, f"alpha={alpha:g} | Validation RMSE={rmse_bp:.6f} bps")

    search = pd.DataFrame(candidates)
    best_index = int(search["validation_rmse_bp"].idxmin())
    selected = dict(candidates[best_index], prediction_unit="basis_points",
                    training_window="Same fixed training set")
    save_csv(folder / "alpha_search.csv", search, index=False)
    model = fitted[best_index]
    # Coefficients use scaled features; convert bps back to returns on save.
    save_csv(folder / "coefficients.csv", pd.DataFrame({
        "feature": columns, "coefficient_bp": model.named_steps["ridge"].coef_,
    }), index=False)
    stage_log(run, folder, f"Selected alpha={selected['alpha']:g}")
    return save_training(run, stage, folder, model, {"train": train, "val": val}, columns, selected)


## 训练

两组分别训练、选参并保存日志。


In [13]:
TRAINERS = {"baseline": train_xgboost, "final": train_ridge}
artifacts = {}  # Save each fitted model before testing
for arm in ARM_LABELS:
    RUN.logger.info("Training arm: %s", ARM_LABELS[arm])
    artifacts[arm] = TRAINERS[arm](
        arm_runs[arm], "holdout", splits["train"], splits["val"], FEATURE_SETS[arm])


20:09:46 | Training arm: Baseline · 6 features + XGBoost
20:09:46 | holdout XGBoost training started | train=6020 / val=751
20:09:46 | Round 1 | train.mae=0.00794250 | val.mae=0.00769099
20:09:46 | Round 2 | train.mae=0.00793992 | val.mae=0.00769065
20:09:46 | Round 3 | train.mae=0.00793818 | val.mae=0.00769117
20:09:46 | Round 4 | train.mae=0.00793714 | val.mae=0.00769132
20:09:46 | Round 5 | train.mae=0.00793559 | val.mae=0.00769062
20:09:46 | Round 6 | train.mae=0.00793472 | val.mae=0.00769040
20:09:46 | Round 7 | train.mae=0.00793358 | val.mae=0.00769004
20:09:46 | Round 8 | train.mae=0.00793189 | val.mae=0.00768945
20:09:46 | Round 9 | train.mae=0.00793042 | val.mae=0.00768872
20:09:46 | Round 10 | train.mae=0.00792943 | val.mae=0.00768911
20:09:46 | Round 11 | train.mae=0.00792800 | val.mae=0.00768969
20:09:46 | Round 12 | train.mae=0.00792578 | val.mae=0.00768951
20:09:46 | Round 13 | train.mae=0.00792464 | val.mae=0.00769002
20:09:46 | Round 14 | train.mae=0.00792285 | val.mae=

## 测试
读取已保存模型，在同一个测试集上分别评价。测试数据不参与轮次、分类阈值或 alpha 的选择。


In [14]:
rows = []
for arm in ARM_LABELS:
    RUN.logger.info("Testing arm: %s", ARM_LABELS[arm])
    metrics, predictions = evaluate_saved(arm_runs[arm], "holdout", artifacts[arm], splits["test"])
    metrics.pop("period", None)
    rows.append(dict(scenario=ARM_LABELS[arm], **metrics))
    save_csv(arm_runs[arm].path / "test_predictions.csv", predictions)
    save_json(arm_runs[arm].path / "summary.json", metrics)
comparison = pd.DataFrame(rows)
save_csv(RUN.path / "comparison_summary.csv", comparison, index=False)
save_json(RUN.path / "status.json", {"status": "completed", "evaluation": "single_chronological_holdout"})


20:09:49 | Testing arm: Baseline · 6 features + XGBoost
20:09:49 | holdout testing started | samples 752 | model loaded from disk
20:09:49 | holdout testing complete | mae_bp=62.3551 | rmse_bp=92.6746
20:09:49 | Testing arm: Final · 6 features + Ridge
20:09:49 | holdout testing started | samples 752 | model loaded from disk
20:09:49 | holdout testing complete | mae_bp=62.3406 | rmse_bp=92.4969


## 同一测试集的结果对比
误差单位为基点，越低越好。MAE 对训练中位数，RMSE 对训练均值；ΔRMSE 为模型减常数。这里只报告这一次测试的指标。


In [15]:
show_table(comparison[["scenario", "n_test", "mae_bp", "rmse_bp", "constant_rmse_bp", "delta_rmse_bp"]])
print("Output directory:", display_path(RUN.path))


,scenario,n_test,mae_bp,rmse_bp,constant_rmse_bp,delta_rmse_bp
0,Baseline · 6 features + XGBoost,752,62.3551,92.6746,92.9958,-0.3212
1,Final · 6 features + Ridge,752,62.3406,92.4969,92.9958,-0.4989


Output directory: output/regression_comparison/20261124_200939_835752


每个方案的 `train_时间戳/` 保存模型、选参记录和训练/验证预测；`test_时间戳/` 保存测试日志与预测。辅助读取、保存逻辑位于 `experiment_utils.py`。


适配使用本地目标和时序验证 MSE；0.002 交易阈值不参与拟合。


## 文献对照

[Chun、Kang 和 Kim (2024)](https://doi.org/10.1186/s40854-024-00648-w) 使用不同样本、目标和特征，仅作设置参考。


In [17]:
SECTION='05_regression'
import json, hashlib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
assert (ROOT/'01_data_preprocessing.ipynb').is_file()
LIT_OUT = ROOT/'output/literature'/SECTION/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
LIT_OUT.mkdir(parents=True)
mlp=ROOT/json.loads((ROOT/'output/fusion/mlp_latest.json').read_text())['path']
run=ROOT/json.loads((mlp/'config.json').read_text())['source_run'];rows=[];INPUTS=[mlp/'config.json']
for name,path in [('Local Regression XGB',run/'baseline/test_predictions.csv'),('Local Ridge',run/'final/test_predictions.csv')]:
    INPUTS.append(path);f=pd.read_csv(path);y=f.target_return.to_numpy();p=f.prediction.to_numpy();mse=np.mean((y-p)**2)
    rows.append({'origin':'local','model':name,'n':len(f),'MSE_percent_squared':mse*10000,'RMSE_bps':np.sqrt(mse)*10000,'MAE_bps':abs(y-p).mean()*10000,'R2_test_mean':1-np.sum((y-p)**2)/np.sum((y-y.mean())**2),'R2_oos_20day_mean':None,'direction_accuracy':np.mean((y>0)==(p>0)),'RMSE_source':'computed from saved predictions'})
for name,mse,r2pct in [('Ridge',1.152,-17.832),('XGB',1.072,-2.093),('MLP1',1.298,-49.676),('MLP2',1.189,-25.527),('MLP3',1.123,-12.049),('MLP4',1.114,-10.307),('MLP5',1.064,-.650)]:
    rows.append({'origin':'paper reported; metric ambiguity','model':'Chun '+name,'n':2348,'paper_MSE_as_labeled':mse,'MSE_percent_squared':None,'RMSE_bps':None,'MAE_bps':None,'R2_test_mean':None,'R2_oos_20day_mean':r2pct/100,'direction_accuracy':None,'RMSE_source':'N/A: not safely derivable from ambiguous MSE label'})
PAPER={'authors':'Dohyun Chun; Jongho Kang; Jihun Kim','year':2024,'doi':'10.1186/s40854-024-00648-w','url':'https://link.springer.com/content/pdf/10.1186/s40854-024-00648-w.pdf','location':'Table 3 panel b PDF p11; Eq5; Table2 note','test':'2012-01-02 to 2020-08-31','unit':'reported R2_oos percent divided by 100; MSE kept as labeled, no RMSE conversion due internal consistency ambiguity','not_reproduced':True}
literature_table=pd.DataFrame(rows)

literature_table.to_csv(LIT_OUT/'literature_comparison.csv',index=False,encoding='utf-8-sig')
(LIT_OUT/'extraction.json').write_text(json.dumps({'checked_on':'2026-11-24','paper':PAPER,'table':literature_table.replace({np.nan:None}).to_dict('records'),'input_sha256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in INPUTS}},ensure_ascii=False,indent=2),encoding='utf-8')
display(literature_table.fillna('N/A'))
print('Saved:',LIT_OUT.relative_to(ROOT))


,origin,model,n,MSE_percent_squared,RMSE_bps,MAE_bps,R2_test_mean,R2_oos_20day_mean,direction_accuracy,RMSE_source,paper_MSE_as_labeled
0,local,Local Regression XGB,752,0.858859,92.674628,62.355094,0.00479,N/A,0.577128,computed from saved predictions,N/A
1,local,Local Ridge,752,0.855567,92.496864,62.340638,0.008604,N/A,0.545213,computed from saved predictions,N/A
2,paper reported; metric ambiguity,Chun Ridge,2348,N/A,N/A,N/A,N/A,-0.17832,N/A,N/A: not safely derivable from ambiguous MSE l...,1.152
3,paper reported; metric ambiguity,Chun XGB,2348,N/A,N/A,N/A,N/A,-0.02093,N/A,N/A: not safely derivable from ambiguous MSE l...,1.072
4,paper reported; metric ambiguity,Chun MLP1,2348,N/A,N/A,N/A,N/A,-0.49676,N/A,N/A: not safely derivable from ambiguous MSE l...,1.298
5,paper reported; metric ambiguity,Chun MLP2,2348,N/A,N/A,N/A,N/A,-0.25527,N/A,N/A: not safely derivable from ambiguous MSE l...,1.189
6,paper reported; metric ambiguity,Chun MLP3,2348,N/A,N/A,N/A,N/A,-0.12049,N/A,N/A: not safely derivable from ambiguous MSE l...,1.123
7,paper reported; metric ambiguity,Chun MLP4,2348,N/A,N/A,N/A,N/A,-0.10307,N/A,N/A: not safely derivable from ambiguous MSE l...,1.114
8,paper reported; metric ambiguity,Chun MLP5,2348,N/A,N/A,N/A,N/A,-0.0065,N/A,N/A: not safely derivable from ambiguous MSE l...,1.064


Saved: output\literature\05_regression\20261124_201044_983083


结论边界：可以讨论小型MLP、树模型与线性模型的表现并不总按复杂度排序；不能比较不同分母的R²，也不能把不同日期、收益目标和特征下的误差差异归因于算法优劣。
